In [ ]:
# 在 Jupyter 里直接运行：随机抽取 >=2 个 sensor 的样本，并 inline 可视化各 sensor 的 plume mask（不保存 PNG）

import os
import csv
import random
import numpy as np
import matplotlib.pyplot as plt

TRAIN_CSV = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/finalDataset/manifest_multisensor_crop_scheme2_train_s5p_replaced_plus_s5p_only_old2025_s5p_balanced_by_plumeid_emit_binary_mask_cleaned_train.csv"
TEST_CSV  = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/finalDataset/manifest_multisensor_crop_scheme2_train_s5p_replaced_plus_s5p_only_old2025_s5p_balanced_by_plumeid_emit_binary_mask_cleaned_test.csv"

SENSORS = ("s2", "l89", "emit", "s5p")
NULL_LIKE = {"", "none", "nan", "null"}

N_SAMPLES = 8
MIN_SENSORS = 2
SEED = 20260410
REQUIRE_FILE_EXISTS = True  # True: 只看文件真实存在的路径

def clean_path(v):
    if v is None:
        return None
    s = str(v).strip()
    return None if s.lower() in NULL_LIKE else s

def read_mask(path):
    # 优先 tifffile，失败再 rasterio
    try:
        import tifffile
        return tifffile.imread(path)
    except Exception:
        import rasterio
        with rasterio.open(path) as ds:
            return ds.read()

def to_2d(arr):
    a = np.asarray(arr)
    if a.ndim == 2:
        return a
    if a.ndim < 2:
        raise ValueError(f"invalid mask ndim={a.ndim}")
    axis = int(np.argmin(a.shape))  # 通常取最小维作为 band/channel 维
    return np.take(a, indices=0, axis=axis)

def collect_candidates(csv_path, split):
    out = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            paths = {}
            for s in SENSORS:
                p = clean_path(row.get(f"{s}_plume_path"))
                if p is None:
                    continue
                if REQUIRE_FILE_EXISTS and (not os.path.exists(p)):
                    continue
                paths[s] = p
            if len(paths) >= MIN_SENSORS:
                out.append({
                    "split": split,
                    "id": row.get("id", ""),
                    "plume_id": row.get("plume_id", ""),
                    "label": row.get("label", ""),
                    "anchor_sensor": row.get("anchor_sensor", ""),
                    "paths": paths,
                })
    return out

cands = collect_candidates(TRAIN_CSV, "train") + collect_candidates(TEST_CSV, "test")
print(f"candidates: {len(cands)}")

rng = random.Random(SEED)
samples = rng.sample(cands, k=min(N_SAMPLES, len(cands)))
print(f"sampled: {len(samples)}")

for i, smp in enumerate(samples, 1):
    sensors = [s for s in SENSORS if s in smp["paths"]]
    fig, axes = plt.subplots(1, len(sensors), figsize=(4.2 * len(sensors), 4.2), squeeze=False)
    axes = axes[0]

    for ax, s in zip(axes, sensors):
        arr = to_2d(read_mask(smp["paths"][s]))
        pos = int((arr > 0).sum())
        ax.imshow(arr, cmap="gray", interpolation="nearest")
        ax.set_title(f"{s} | pos={pos}")
        ax.axis("off")

    fig.suptitle(
        f"[{i}/{len(samples)}] {smp['split']} | id={smp['id']} | plume_id={smp['plume_id']} | "
        f"label={smp['label']} | anchor={smp['anchor_sensor']}",
        fontsize=11
    )
    plt.tight_layout()
    plt.show()
